# Solution to p16

Let `P` be a permutation matrix. Relabeling receivers and senders gives `A'=PAP^T`, while relabeling feature rows gives `X'=PX`. Since `P^T P=I`,

`A'X'=(PAP^T)(PX)=PA(P^T P)X=PAX=P(AX)`.

Thus sum aggregation is permutation-equivariant: permuting the input nodes permutes output rows the same way.

For mean aggregation, let `1` be the all-ones vector and `d=A1`, with positive entries, and `D=diag(d)`. A permutation preserves `1`, so `d'=A'1=PAP^T1=PA1=Pd`, hence `D'=PDP^T` and `(D')^{-1}=PD^{-1}P^T`. Therefore

`(D')^{-1}A'X' = PD^{-1}P^T PAP^T PX = P(D^{-1}AX)`.

A shared row-wise linear projection has form `XW+1b^T`. Because it uses the same `W,b` for every row, applying it before or after `P` gives the same permuted result. In contrast, appending an absolute feature such as the original node index `(0,1,...,N-1)` without permuting that feature with the nodes assigns meaning to storage position and breaks equivariance.

### Answer check

In [ ]:
import torch

A=torch.tensor([[1.,1.,0.],[0.,1.,1.],[1.,0.,1.]])
X=torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
P=torch.tensor([[0.,0.,1.],[1.,0.,0.],[0.,1.,0.]])
Ap=P@A@P.T; Xp=P@X
torch.testing.assert_close(Ap@Xp,P@(A@X))
D=torch.diag(A.sum(1)); Dp=torch.diag(Ap.sum(1))
torch.testing.assert_close(torch.linalg.solve(Dp,Ap@Xp),P@torch.linalg.solve(D,A@X))
W=torch.tensor([[1.,-1.],[2.,3.]])
b=torch.tensor([.5,-.5])
torch.testing.assert_close((P@X)@W+b,P@(X@W+b))
assert all((
    torch.allclose(Ap@Xp,P@(A@X),atol=1e-6,rtol=1e-6),
    torch.allclose(torch.linalg.solve(Dp,Ap@Xp),P@torch.linalg.solve(D,A@X),atol=1e-6,rtol=1e-6),
    torch.allclose((P@X)@W+b,P@(X@W+b),atol=1e-6,rtol=1e-6),
))
